In [2]:
from DWG_to_DuckDB_ETL import *
import os
import shutil
import numpy as np
import math

dwg_folder = r"C:\Users\Usuario\Desktop\projetos\paiva"
temp_folder = r"C:\Users\Usuario\Desktop\projetos\paiva\temp"
output_folder = dwg_folder

os.makedirs(temp_folder, exist_ok=True)
dwg_file = os.path.join(dwg_folder, "PV Paiva - Eletromecânico.dwg")

shutil.copy(dwg_file, temp_folder)

convert_dwg_to_dxf(temp_folder, output_folder)

dxf_file = os.path.join(output_folder, "PV Paiva - Eletromecânico.dxf")

temp_dwg_file = os.path.join(temp_folder, "PV Paiva - Eletromecânico.dwg")

data = extract_data_from_dxf(dxf_file)
print("Data Loaded")

os.remove(dxf_file)
os.remove(temp_dwg_file)

Data Loaded


In [3]:
ELM_df = pd.DataFrame(data)

print(ELM_df.describe())
print(ELM_df.info())
print(ELM_df['name'].value_counts())

                  X              Y  rotation
count   7396.000000    7396.000000    7396.0
mean   35046.123692  128103.163453       0.0
std      598.040467     966.183745       0.0
min    33667.550000  126018.800025       0.0
25%    34601.690000  127315.224625       0.0
50%    34979.686308  128157.605775       0.0
75%    35464.785992  128982.236025       0.0
max    36625.890000  129774.083025       0.0
<class 'pandas.DataFrame'>
RangeIndex: 7396 entries, 0 to 7395
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   type      7396 non-null   str    
 1   layer     7396 non-null   str    
 2   handle    7396 non-null   str    
 3   name      7396 non-null   str    
 4   X         7396 non-null   float64
 5   Y         7396 non-null   float64
 6   rotation  7396 non-null   int64  
dtypes: float64(2), int64(1), str(4)
memory usage: 780.2 KB
None
name
EMF_Tracker_1PX64_South    3025
EMF_Tracker_1PX64_North    1946
EMF_Tracker_1PX

In [6]:
mask = (ELM_df['name'] == 'EMF_Tracker_1PX32_North') | (ELM_df['name'] == 'EMF_Tracker_1PX64_North') | (ELM_df['name'] == 'EMF_Tracker_1PX32_South') | (ELM_df['name'] == 'EMF_Tracker_1PX64_South')
tracker_df = ELM_df[mask].copy()

tracker_df.drop(columns=['handle', 'type', 'rotation'], inplace=True)

print(tracker_df)
print(tracker_df.describe())
print(tracker_df.info())
print(tracker_df['name'].value_counts())

                   layer                     name             X              Y
0     EMF_em_tracker_32p  EMF_Tracker_1PX32_North  35794.290000  129434.080025
1     EMF_em_tracker_64p  EMF_Tracker_1PX64_North  35926.591051  129114.316025
2     EMF_em_tracker_64p  EMF_Tracker_1PX64_North  35920.291051  129114.316025
3     EMF_em_tracker_64p  EMF_Tracker_1PX64_North  35913.991051  129114.316025
4     EMF_em_tracker_64p  EMF_Tracker_1PX64_North  35945.491051  129070.814965
...                  ...                      ...           ...            ...
7391  EMF_em_tracker_32p  EMF_Tracker_1PX32_South  34670.986150  126149.510111
7392  EMF_em_tracker_32p  EMF_Tracker_1PX32_South  34677.286150  126149.490602
7393  EMF_em_tracker_32p  EMF_Tracker_1PX32_South  34664.686150  126106.022575
7394  EMF_em_tracker_32p  EMF_Tracker_1PX32_South  34670.986150  126106.007111
7395  EMF_em_tracker_32p  EMF_Tracker_1PX32_South  34677.286150  126105.987602

[7396 rows x 4 columns]
                  X        

In [10]:
def group_structures(df, max_distance=66):
    """
    Groups adjacent structures in pairs (2) while respecting:
    - TAG boundaries
    - Maximum distance constraint (22m default)
    - Sorted by Position Y then X
    """
    # Sort by Position Y (primary) and Position X (secondary)
    df['X'] = round(df['X'],2)
    df['Y'] = round(df['Y'],2)
    df = df.sort_values(['X', 'Y'], ascending=[True, False]).reset_index(drop=True)

    # Calculate pairwise distances between consecutive structures
    df['x_diff'] = round(df['X'].diff(),2)
    df['y_diff'] = round(df['Y'].diff(),2)
    df['distance'] = np.sqrt(df['x_diff']**2 + df['y_diff']**2)

    # Helper function to check if it's a 64p/32p pair

    def is_64_32_pair(current_name, prev_name):
        if pd.isna(prev_name):
            return False
        has_64p = '64p' in str(current_name) or '64p' in str(prev_name)
        has_32p = '32p' in str(current_name) or '32p' in str(prev_name)
        return has_64p and has_32p and current_name != prev_name

    # Create group boundaries where:
    # 1. X is different
    # 2. Distance > max_distance OR
    # 3. It's the first structure
    # 4. One structure is 64p and the other is 32p
    df['group_boundary'] = (
        (df['x_diff'] > 0.1) |
        (abs(df['y_diff']) > max_distance) |
        (df.index == 0) |
        (df.apply(lambda row: is_64_32_pair(row['name'], df.loc[row.name-1,'name'] if row.name > 0 else None), axis=1))
    )
    # Create group IDs
    df['group_id'] = df['group_boundary'].cumsum()

    # Split into groups
    result_groups = []

    for _, group_df in df.groupby('group_id'):
        # Split groups larger than 2 into pairs
        size = len(group_df)
        if size <= 2:
            result_groups.append(group_df)

            continue

        n_pairs = size // 2
        for i in range(n_pairs):
            result_groups.append(group_df.iloc[i*2:i*2+2])


        if size % 2:  # Handle odd-sized groups
            result_groups.append(group_df.iloc[-1:])
    df.to_excel(r"C:\Users\Usuario\Desktop\teste.xlsx")
    return result_groups

def create_group_positions(groups):
    """Create the output DataFrame from groups (same as before)"""
    group_data = []
    for i, group_df in enumerate(groups, 1):
        if len(group_df) > 1:
            first_struct = group_df.iloc[0]
            second_struct = group_df.iloc[1]
            group_data.append({
                'Group_ID': i,
                'X': first_struct['X'],
                'Y': first_struct['Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer'],
                'Layer-2': second_struct['layer'],
                'Name-1': first_struct['name'],
                'Name-2': second_struct['name']
            })
        else:
            first_struct = group_df.iloc[0]
            group_data.append({
                'Group_ID': i,
                'X': first_struct['X'],
                'Y': first_struct['Y'],
                'Group_Size': len(group_df),
                'Layer-1': first_struct['layer'],
                'Name-1': first_struct['name']
            })
    return pd.DataFrame(group_data)

In [11]:
results = group_structures(tracker_df)
group_positions = create_group_positions(results)
group_positions.to_excel(r"C:\Users\Usuario\Desktop\teste1.xlsx")
print(group_positions['Group_Size'].value_counts())
print(group_positions)

Group_Size
1    3988
2    1704
Name: count, dtype: int64
      Group_ID         X          Y  Group_Size             Layer-1  \
0            1  33667.55  129034.80           2  EMF_em_tracker_32p   
1            2  33667.55  128909.06           1  EMF_em_tracker_64p   
2            3  33667.55  128823.02           1  EMF_em_tracker_64p   
3            4  33673.85  129034.80           2  EMF_em_tracker_32p   
4            5  33673.85  128909.06           1  EMF_em_tracker_64p   
...        ...       ...        ...         ...                 ...   
5687      5688  36600.69  129536.32           1  EMF_em_tracker_32p   
5688      5689  36606.99  129536.32           1  EMF_em_tracker_32p   
5689      5690  36613.29  129536.32           1  EMF_em_tracker_32p   
5690      5691  36619.59  129536.32           1  EMF_em_tracker_32p   
5691      5692  36625.89  129536.32           1  EMF_em_tracker_32p   

                 Layer-2                   Name-1                   Name-2  
0     EMF_em_

In [13]:
correction_position_y_32_first = 42.7655
correction_position_y_64_first = 21.5015

group_positions['Script'] = np.where(
    group_positions['Group_Size'] == 1,
    group_positions.apply(
        lambda row: (
            f"._insert {row['Name-1']} {row['X']},{row['Y']} 1 1 0"
        ),
    axis=1
    ),
    np.where(
        group_positions['Layer-1'] == 'EMF_em_tracker_32p',
        group_positions.apply(
            lambda row: (
                f"._insert EMF_tracker_96p {row['X']},{row['Y']-correction_position_y_32_first} 1 1 0"
            ),
        axis=1
        ),
        group_positions.apply(
            lambda row: (
                f"._insert EMF_tracker_96p {row['X']},{row['Y']-correction_position_y_64_first} 1 1 0"
            ),
        axis=1
        )
    )
)

print(group_positions)

group_positions.to_excel(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Separador de Strings\teste.xlsx")
with open(r"C:\Users\Usuario\Desktop\Pessoal\Python\Ecotechne\Reinserting Trackers\Reinserting Trackers 1st+2str.scr", 'w') as f:
    for i in group_positions['Layer-1'].unique():
        mask = group_positions['Layer-1'] == i

        if mask.any():
            f.write(f'(command-s "._layer" "set" "{i}" "")\n')
            f.write('\n'.join(group_positions[mask]['Script'].astype(str)) + '\n')

      Group_ID         X          Y  Group_Size             Layer-1  \
0            1  33667.55  129034.80           2  EMF_em_tracker_32p   
1            2  33667.55  128909.06           1  EMF_em_tracker_64p   
2            3  33667.55  128823.02           1  EMF_em_tracker_64p   
3            4  33673.85  129034.80           2  EMF_em_tracker_32p   
4            5  33673.85  128909.06           1  EMF_em_tracker_64p   
...        ...       ...        ...         ...                 ...   
5687      5688  36600.69  129536.32           1  EMF_em_tracker_32p   
5688      5689  36606.99  129536.32           1  EMF_em_tracker_32p   
5689      5690  36613.29  129536.32           1  EMF_em_tracker_32p   
5690      5691  36619.59  129536.32           1  EMF_em_tracker_32p   
5691      5692  36625.89  129536.32           1  EMF_em_tracker_32p   

                 Layer-2                   Name-1                   Name-2  \
0     EMF_em_tracker_32p  EMF_Tracker_1PX32_North  EMF_Tracker_1PX32_